# M4-B: Skeleton-Transformer V2 Behaviour Classifier
This notebook trains the optimal sequence classification model for the Smart Surveillance System. It takes 16-frame keypoint sequences extracted via YOLOv8, computes 283-dimensional engineered features (bones, angles, velocity, acceleration), and processes them through a Transformer Encoder.

**Features included:**
1. Translation/Scale invariant feature engineering.
2. Staged Training (Head first, then full body).
3. Exponential Moving Average (EMA) for weight smoothing.
4. Skeleton Mixup augmentation.

In [5]:
import os, cv2, gc, random, time, sys, math, copy
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from sklearn.metrics import accuracy_score, f1_score
from pathlib import Path
from tqdm.auto import tqdm

# ==========================================
# 1. Configuration Setup
# ==========================================
# Update this path if your extracted clips are located elsewhere
SKEL_ROOT        = r'../M4B_skeleton_clips_v2'  
TARGET_CLASSES   = ['Violence', 'Threat', 'Normal']

SKEL_CLIP_FRAMES = 16
MAX_PERSONS      = 2
N_KEYPOINTS      = 17

# Hyperparameters
EPOCHS        = 120
BATCH         = 32
LR_HEAD       = 1e-3     
LR_FULL       = 5e-5     
FREEZE_EPOCHS = 10       
PATIENCE      = 30       
EMA_DECAY     = 0.998    
MIXUP_ALPHA   = 0.3      

SAVE_PATH = r'best_transformer_v2.pt'
SAVE_EMA  = r'best_transformer_v2_ema.pt'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Hardware utilized: {DEVICE}")

Hardware utilized: cuda


## 2. Advanced Feature Extraction Pipeline
Calculates spatial and temporal dynamics explicitly so the Transformer doesn't waste capacity learning basic physics.

In [6]:
BONES = [(5,7),(7,9),(6,8),(8,10),(5,6),(11,13),(13,15),(12,14),(14,16),(11,12),(5,11),(6,12),(0,5),(0,6),(0,1),(0,2)]
ANGLE_TRIPLETS = [(5,7,9),(6,8,10),(11,13,15),(12,14,16),(5,0,6)]

def compute_bones(kps):
    out = []
    for j1, j2 in BONES:
        dx, dy = kps[j1,0]-kps[j2,0], kps[j1,1]-kps[j2,1]
        out.append(math.sqrt(dx*dx + dy*dy + 1e-8))
    return np.array(out, dtype=np.float32)

def compute_angles(kps):
    out = []
    for a, v, b in ANGLE_TRIPLETS:
        va, vb = kps[a]-kps[v], kps[b]-kps[v]
        c = np.dot(va, vb) / (np.linalg.norm(va)*np.linalg.norm(vb) + 1e-8)
        out.append(math.acos(np.clip(c, -1, 1)) / math.pi)
    return np.array(out, dtype=np.float32)

def compute_inter(k1, k2):
    c1, c2 = k1[:,:2].mean(0), k2[:,:2].mean(0)
    cd = np.linalg.norm(c1-c2)
    md = np.linalg.norm(k1[:,:2][:,None]-k2[:,:2][None,:], axis=2).min()
    hh = min(np.linalg.norm(k1[9,:2]-k2[0,:2]), np.linalg.norm(k1[10,:2]-k2[0,:2]),
             np.linalg.norm(k2[9,:2]-k1[0,:2]), np.linalg.norm(k2[10,:2]-k1[0,:2]))
    return np.array([cd, md, hh], dtype=np.float32)

class TransformerSkeletonDataset(Dataset):
    def __init__(self, root, split, classes, augment=False, feat_mean=None, feat_std=None):
        self.samples = []
        self.augment = augment
        self.feat_mean = feat_mean
        self.feat_std = feat_std
        base = Path(root) / split
        
        if base.exists():
            for ci, cls in enumerate(classes):
                d = base / cls
                if d.exists():
                    for p in d.glob('*.npy'):
                        self.samples.append((p, ci))
            random.shuffle(self.samples)
        else:
            print(f"Warning: Dataset path {base} not found. Please verify SKEL_ROOT.")

    def __len__(self):
        return len(self.samples)

    def _mask_low_conf(self, clip):
        for p in range(MAX_PERSONS):
            valid = clip[:, p, :, 2] > 0.3
            for j in range(N_KEYPOINTS):
                if valid[:, j].sum() > 0:
                    mp = clip[valid[:, j], p, j, :2].mean(0)
                else:
                    mp = np.array([0.5, 0.5])
                for t in range(clip.shape[0]):
                    if not valid[t, j]:
                        clip[t, p, j, :2] = mp
                        clip[t, p, j, 2] = 0.0
        return clip

    def _build(self, clip):
        T = clip.shape[0]
        clip = self._mask_low_conf(clip)
        rows = []
        for t in range(T):
            f = []
            for p in range(MAX_PERSONS):
                k = clip[t, p]
                f.extend([k[:,:2].flatten(), k[:,2], compute_bones(k), compute_angles(k)])
            f.append(compute_inter(clip[t,0], clip[t,1]))
            rows.append(np.concatenate(f))
        
        feat = np.stack(rows)
        psize = 34+17+16+5
        
        # Calculate Velocity and Acceleration
        for p in range(MAX_PERSONS):
            s = p * psize
            pos = feat[:, s:s+34]
            vel = np.zeros_like(pos); vel[1:] = pos[1:]-pos[:-1]
            acc = np.zeros_like(vel); acc[2:] = vel[2:]-vel[1:-1]
            feat = np.concatenate([feat, vel, acc], axis=1)
        return feat

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        clip = np.load(str(path)).copy()
        
        # Data Augmentation (Flipping, Scaling, Translating)
        if self.augment:
            if random.random() > 0.5:
                clip[:,:,:,0] = 1.0 - clip[:,:,:,0]
                for l, r in [(1,2),(3,4),(5,6),(7,8),(9,10),(11,12),(13,14),(15,16)]:
                    clip[:,:,[l,r]] = clip[:,:,[r,l]]
            dx, dy = random.uniform(-0.05,0.05), random.uniform(-0.05,0.05)
            clip[:,:,:,0] = np.clip(clip[:,:,:,0]+dx, 0, 1)
            clip[:,:,:,1] = np.clip(clip[:,:,:,1]+dy, 0, 1)
            s = random.uniform(0.85, 1.15)
            for p in range(MAX_PERSONS):
                cx, cy = clip[:,p,:,0].mean(), clip[:,p,:,1].mean()
                clip[:,p,:,0] = np.clip((clip[:,p,:,0]-cx)*s+cx, 0, 1)
                clip[:,p,:,1] = np.clip((clip[:,p,:,1]-cy)*s+cy, 0, 1)
                
        feat = self._build(clip)
        
        # Standardization
        if self.feat_mean is not None:
            feat = (feat - self.feat_mean) / (self.feat_std + 1e-8)
            
        return torch.tensor(feat, dtype=torch.float32), label

## 3. Model Architecture & Regularization Utilities

In [7]:
class SkeletonTransformer(nn.Module):
    def __init__(self, input_dim, num_classes=3, d_model=128,
                 nhead=4, num_layers=4, dim_ff=256, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout * 0.3),
        )
        self.pos_embed = nn.Parameter(torch.randn(1, SKEL_CLIP_FRAMES, d_model) * 0.02)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            dropout=dropout, batch_first=True, activation='gelu',
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(d_model)
        
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(d_model // 2, num_classes),
        )

    def forward(self, x):
        B = x.shape[0]
        x = self.input_proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        pos = torch.cat([torch.zeros(1, 1, x.shape[2], device=x.device),
                         self.pos_embed], dim=1)
        x = x + pos
        x = self.transformer(x)
        x = self.norm(x[:, 0]) # Extract CLS token output
        return self.head(x)

class EMA:
    def __init__(self, model, decay=0.998):
        self.decay = decay
        self.shadow = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.shadow[name] = self.decay * self.shadow[name] + (1-self.decay) * param.data

    def apply(self, model):
        backup = {}
        for name, param in model.named_parameters():
            if name in self.shadow:
                backup[name] = param.data.clone()
                param.data = self.shadow[name].clone()
        return backup

    def restore(self, model, backup):
        for name, param in model.named_parameters():
            if name in backup:
                param.data = backup[name]

def skeleton_mixup(x, y, alpha=0.3):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1 - lam)
    index = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam

## 4. Main Training Execution

In [8]:
# 1. Compute feature statistics for standardization
print('Computing dataset statistics for Standard Scaler...')
tmp_ds = TransformerSkeletonDataset(SKEL_ROOT, 'train', TARGET_CLASSES)

if len(tmp_ds) == 0:
    print("CRITICAL: No training data found. Execution halted.")
else:
    samples = [tmp_ds[i][0].numpy() for i in range(min(2000, len(tmp_ds)))]
    all_feat = np.concatenate(samples, axis=0)
    FEAT_MEAN, FEAT_STD = all_feat.mean(axis=0), all_feat.std(axis=0)
    FEAT_DIM = FEAT_MEAN.shape[0]
    del tmp_ds, samples, all_feat
    print(f'Feature dimension per frame calculated as: {FEAT_DIM}')

    # 2. Setup DataLoaders
    train_ds = TransformerSkeletonDataset(SKEL_ROOT, 'train', TARGET_CLASSES, augment=True, feat_mean=FEAT_MEAN, feat_std=FEAT_STD)
    val_ds = TransformerSkeletonDataset(SKEL_ROOT, 'val', TARGET_CLASSES, augment=False, feat_mean=FEAT_MEAN, feat_std=FEAT_STD)
    
    train_ld = DataLoader(train_ds, batch_size=BATCH, shuffle=True, pin_memory=True)
    val_ld = DataLoader(val_ds, batch_size=BATCH, shuffle=False, pin_memory=True)

    # 3. Model & EMA Initialization
    model = SkeletonTransformer(FEAT_DIM).to(DEVICE)
    ema = EMA(model, decay=EMA_DECAY)
    criterion = nn.CrossEntropyLoss()
    
    # 4. Stage 1 Setup: Freeze Body
    body_params = [p for n, p in model.named_parameters() if 'head' not in n]
    head_params = [p for n, p in model.named_parameters() if 'head' in n]
    for p in body_params: p.requires_grad = False
        
    optimizer = optim.AdamW(head_params, lr=LR_HEAD, weight_decay=1e-2)
    scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2, eta_min=1e-6)

    best_f1, best_f1_ema, pat = 0.0, 0.0, 0

    print('\nStarting Staged Training...')
    for epoch in range(1, EPOCHS + 1):
        
        # Transition to Stage 2: Unfreeze Body
        if epoch == FREEZE_EPOCHS + 1:
            print(f'\n>>> UNFREEZING transformer body (lr={LR_FULL}) <<<\n')
            for p in body_params: p.requires_grad = True
            optimizer = optim.AdamW(model.parameters(), lr=LR_FULL, weight_decay=1e-2)
            scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=15, T_mult=2, eta_min=1e-7)

        # --- Training Loop ---
        model.train()
        tloss = 0.0
        loop = tqdm(train_ld, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
        for clips, labels in loop:
            clips, labels = clips.to(DEVICE), labels.to(DEVICE)
            
            if epoch > FREEZE_EPOCHS and MIXUP_ALPHA > 0:
                clips_mix, labels_a, labels_b, lam = skeleton_mixup(clips, labels, MIXUP_ALPHA)
                optimizer.zero_grad()
                out = model(clips_mix)
                loss = lam * criterion(out, labels_a) + (1-lam) * criterion(out, labels_b)
            else:
                optimizer.zero_grad()
                out = model(clips)
                loss = criterion(out, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            ema.update(model)
            
            tloss += loss.item()
            loop.set_postfix(loss=loss.item())

        tloss /= len(train_ld)
        scheduler.step()

        # --- Validation Loop (Standard Weights) ---
        model.eval()
        yt, yp = [], []
        with torch.no_grad():
            for clips, labels in val_ld:
                preds = model(clips.to(DEVICE)).argmax(1).cpu().numpy()
                yt.extend(labels.numpy()); yp.extend(preds)
        vf1 = f1_score(yt, yp, average='weighted', zero_division=0)

        # --- Validation Loop (EMA Weights) ---
        backup = ema.apply(model)
        yt_e, yp_e = [], []
        with torch.no_grad():
            for clips, labels in val_ld:
                preds = model(clips.to(DEVICE)).argmax(1).cpu().numpy()
                yt_e.extend(labels.numpy()); yp_e.extend(preds)
        vf1_ema = f1_score(yt_e, yp_e, average='weighted', zero_division=0)
        ema.restore(model, backup)

        # --- Checkpoint Handling ---
        improved = ''
        if vf1 > best_f1:
            best_f1, pat, improved = vf1, 0, ' [Standard Save]'
            torch.save(model.state_dict(), SAVE_PATH)
        else:
            pat += 1

        if vf1_ema > best_f1_ema:
            best_f1_ema = vf1_ema
            backup = ema.apply(model)
            torch.save(model.state_dict(), SAVE_EMA)
            ema.restore(model, backup)
            improved += ' [EMA Save]'

        stage = 'HEAD' if epoch <= FREEZE_EPOCHS else 'FULL'
        print(f'Ep {epoch:>3} [{stage}] | Loss: {tloss:.4f} | F1: {vf1:.4f} | EMA F1: {vf1_ema:.4f}{improved}')

        # --- Early Stopping ---
        if pat >= PATIENCE and epoch > FREEZE_EPOCHS + 10:
            print(f'\nEarly stop triggered at epoch {epoch}')
            break

    print(f'\nTraining Complete. Best Standard F1: {best_f1:.4f} | Best EMA F1: {best_f1_ema:.4f}')

Computing dataset statistics for Standard Scaler...
Feature dimension per frame calculated as: 283


C:\Users\jingy\AppData\Local\Temp\ipykernel_23604\448313139.py:18: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)



Starting Staged Training...


Ep   1 [HEAD] | Loss: 0.8788 | F1: 0.5647 | EMA F1: 0.5574 [Standard Save] [EMA Save]


Ep   2 [HEAD] | Loss: 0.8244 | F1: 0.5504 | EMA F1: 0.5732 [EMA Save]


...

Ep  27 [FULL] | Loss: 0.6454 | F1: 0.4831 | EMA F1: 0.4760


Ep  28 [FULL] | Loss: 0.6530 | F1: 0.4828 | EMA F1: 0.4715


Ep  29 [FULL] | Loss: 0.6429 | F1: 0.4511 | EMA F1: 0.4758


Ep  30 [FULL] | Loss: 0.6478 | F1: 0.4515 | EMA F1: 0.4656


Ep  31 [FULL] | Loss: 0.6553 | F1: 0.4800 | EMA F1: 0.4671

Early stop triggered at epoch 31

Training Complete. Best Standard F1: 0.5647 | Best EMA F1: 0.5732
